# Prompt Testing and Iteration

When a prompt does not behave the way you want, it is tempting to keep tweaking it and rerunning a single example until it looks right. That approach hides problems: a change that fixes one case often quietly breaks another.

This notebook takes a more reliable approach: you build a small set of test cases, run each prompt variant against all of them, and compare the results, so you can see what a change actually does before you commit to it. You will follow this loop throughout:

```mermaid
flowchart LR
    A[Task] --> B[Test cases]
    B --> C[Prompt variant]
    C --> D[Run chain]
    D --> E[Inspect outputs]
    E --> F[Revise prompt]
    F --> C
```

## Learning Objectives

By the end of this notebook, you should be able to:

- Build a small evaluation set that exposes a prompt's weak spots.
- Compare a baseline prompt against a stricter one on the same cases.
- Apply few-shot examples to improve consistency on harder cases.
- Reuse this prompt-testing loop in your own projects.

In [1]:
# Keep the setup cell focused so the notebook is easy to rerun from the top.
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate


def print_outputs(cases: list[dict[str, str]], outputs: list[str]) -> None:
    for case, output in zip(cases, outputs):
        print("=" * 100)
        print(f"Case: {case['label']}")
        print(f"What to watch: {case['watch_for']}")
        print(output)

In [2]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=512,
)

output_parser = StrOutputParser()

## Step 1 - Create a Tiny Evaluation Set

For prompt testing, a tiny but varied test set is often more useful than one perfect example. The goal is to include a few cases that are different enough to expose prompt weaknesses.

In this notebook, we will use a simple support-triage task. Each case is a short customer message, and we want the model to:
- estimate urgency
- identify the topic
- suggest a short response


In [3]:
# Keep the test set small enough to inspect manually.
support_cases = [
    {
        "label": "Damaged package",
        "message": "My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.",
        "watch_for": "Should likely be high urgency and mention replacement or support steps.",
    },
    {
        "label": "Password reset",
        "message": "I cannot sign in to my account because I forgot my password. How do I reset it?",
        "watch_for": "Should be account-related and not overstate urgency.",
    },
    {
        "label": "Double charge",
        "message": "I think my card was charged twice for the same order. Can someone check this?",
        "watch_for": "Should identify billing and treat it as fairly urgent.",
    },
    {
        "label": "Sustainability question",
        "message": "Your website says your products are sustainable. What materials do you actually use?",
        "watch_for": "Should stay low urgency and answer as a product question, not a complaint.",
    },
]

support_cases

[{'label': 'Damaged package',
  'message': 'My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.',
  'watch_for': 'Should likely be high urgency and mention replacement or support steps.'},
 {'label': 'Password reset',
  'message': 'I cannot sign in to my account because I forgot my password. How do I reset it?',
  'watch_for': 'Should be account-related and not overstate urgency.'},
 {'label': 'Double charge',
  'message': 'I think my card was charged twice for the same order. Can someone check this?',
  'watch_for': 'Should identify billing and treat it as fairly urgent.'},
 {'label': 'Sustainability question',
  'message': 'Your website says your products are sustainable. What materials do you actually use?',
  'watch_for': 'Should stay low urgency and answer as a product question, not a complaint.'}]

A compact evaluation set like this helps you ask practical questions:
- Which cases produce messy or inconsistent output?
- Which cases are misclassified?
- Which instructions are too vague?

Notice that the test set mixes logistics, account access, billing, and product questions. That variety makes the prompt easier to stress-test.

## Step 2 - Start With a Baseline Prompt

We begin with a deliberately simple prompt. It is good enough to run, but it leaves a lot of room for the model to choose its own output structure.

In [4]:
# This baseline prompt defines the task but does not strongly constrain the format.
baseline_template = """You are helping an ecommerce support team.
Read the customer message and identify the urgency, the topic, and a short reply.

Customer message:
{message}
"""

baseline_prompt = PromptTemplate(
    template=baseline_template,
    input_variables=["message"],
)

baseline_chain = baseline_prompt | llm | output_parser

In [5]:
# Try the baseline prompt on one case before scaling up.
print(baseline_chain.invoke({"message": support_cases[0]["message"]}))

**Urgency:** High – needs a replacement before Friday.  
**Topic:** Damaged mug, replacement request.  

**Reply:**  
"I'm sorry the mug arrived broken. We'll ship a replacement to you immediately with expedited delivery so it arrives before Friday. Thank you for your patience!"


## Step 3 - Evaluate the Baseline Across All Cases

One output is not enough to judge a prompt. Prompt testing becomes more informative when you run the same prompt on several cases and compare the outputs side by side.

In [6]:
# batch() lets us reuse the same prompt on many cases at once.
baseline_inputs = [{"message": case["message"]} for case in support_cases]
baseline_outputs = baseline_chain.batch(baseline_inputs)
print_outputs(support_cases, baseline_outputs)

Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
**Urgency:** High  
**Topic:** Damaged item – request for a replacement mug  
**Reply:**  
"I'm sorry the mug arrived broken. We'll ship a replacement to you immediately with expedited delivery so it arrives before Friday. Let me know if there's anything else I can help with."
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
**Urgency:** Low  
**Topic:** Password reset / account login  

**Reply:**  
Hi there! To reset your password, click the “Forgot password?” link on the sign‑in page, enter the email address linked to your account, and follow the instructions in the email you receive. If you don’t see the email, check your spam/junk folder or let us know and we’ll help you further.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
**Urgency:** Moderate  
**Topic:** Duplicate charge / billing issue

### What Usually Goes Wrong in the Baseline

With a loose prompt, common issues include:
- inconsistent formatting from one case to another
- uneven detail across cases
- vague urgency labels
- responses that are friendly but hard to compare systematically

This is a good sign that we need a stronger prompt specification, not necessarily a different model.

## Step 4 - Improve the Prompt With Stronger Instructions

Now we make the task more explicit. We still use the same model, but we add tighter rules for the output format so the results become easier to inspect and compare.

In [7]:
# Stronger instructions often improve consistency more than changing the model does.
improved_template = """You are triaging ecommerce support messages.
Analyze the customer message and return exactly this format:

Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Keep the reply concise and practical.

Customer message:
{message}
"""

improved_prompt = PromptTemplate(
    template=improved_template,
    input_variables=["message"],
)

improved_chain = improved_prompt | llm | output_parser

In [8]:
# Compare the improved prompt on the same evaluation set.
improved_outputs = improved_chain.batch(baseline_inputs)
print_outputs(support_cases, improved_outputs)

Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
Urgency: high  
Topic: product  
Suggested reply:  
- I'm sorry to hear the mug arrived broken; we can send a replacement right away.  
- Please let us know your order number and we’ll ship the new mug to you before Friday.
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
Urgency: low  
Topic: account  
Suggested reply:  
- Click the “Forgot password” link on the login page and follow the instructions sent to your email.  
- If you don’t receive the email, let us know and we’ll help you reset it.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
Urgency: medium  
Topic: billing  
Suggested reply:  
- Thank you for reaching out; we’ll review your account for duplicate charges right away.  
- Please allow 24 hours for us to investigate and we’ll update you with the outcome.
Case: Sustainability questio

#### Compare Two Prompt Variants on One Hard Case

Comparing full test sets is useful, but sometimes you also want to zoom in on one case that feels tricky and inspect how each prompt handles it.

In [9]:
hard_case = support_cases[2]["message"]

print("BASELINE")
print(baseline_chain.invoke({"message": hard_case}))
print("\n" + "-" * 100 + "\n")
print("IMPROVED")
print(improved_chain.invoke({"message": hard_case}))

BASELINE
**Urgency:** High  
**Topic:** Duplicate charge / Payment issue  

**Reply:**  
Hi! I’m sorry you’re seeing a duplicate charge. Could you please share your order number so we can look into it right away? We’ll sort this out for you as quickly as possible.

----------------------------------------------------------------------------------------------------

IMPROVED
Urgency: medium  
Topic: billing  
Suggested reply:  
- I'm sorry to hear about the duplicate charge; I'll look into it right away.  
- Please allow 24 hours for us to confirm the issue and issue a refund if needed.


## Step 5 - Add Few-Shot Examples for Harder Edge Cases

If the output is still inconsistent, few-shot examples can teach the model the exact style you want. This is especially useful when the format matters or when the task includes edge cases that the model may interpret differently.

```mermaid
flowchart LR
    A[Example input-output pairs] --> B[FewShotPromptTemplate]
    C[New customer message] --> B
    B --> D[Formatted prompt]
    D --> E[More controlled output]
```

The examples do not retrain the model. They just give the model a clearer pattern to imitate for the current request.

In [10]:
# Each example demonstrates the output style we want on a realistic support case.
examples = [
    {
        "message": "My order still has not arrived and I need it tomorrow for an event.",
        "analysis": "Urgency: high\nTopic: shipping\nSuggested reply:\n- I am sorry your order has not arrived in time for your event.\n- Please send us your order number so we can check shipping options right away.",
    },
    {
        "message": "I forgot my password and cannot get into my account.",
        "analysis": "Urgency: medium\nTopic: account\nSuggested reply:\n- Please use the password reset link on the sign-in page to create a new password.\n- If that does not work, contact support and we can help verify your account.",
    },
]

example_prompt = PromptTemplate(
    input_variables=["message", "analysis"],
    template="""Example message:\n{message}\n\nExample output:\n{analysis}""",
)

prefix = """You are triaging ecommerce support messages.
Return exactly this format:
Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Use the examples below as style guidance."""

suffix = """Now analyze this customer message:
{message}"""

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)

few_shot_chain = few_shot_prompt | llm | output_parser

#### Inspect the Final Few-Shot Prompt

When working with more complex prompt templates, printing the final prompt is a great debugging habit. It lets you check whether the examples and the new input were assembled the way you expected.

In [11]:
print(few_shot_prompt.format(message=support_cases[0]["message"]))

You are triaging ecommerce support messages.
Return exactly this format:
Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Use the examples below as style guidance.

Example message:
My order still has not arrived and I need it tomorrow for an event.

Example output:
Urgency: high
Topic: shipping
Suggested reply:
- I am sorry your order has not arrived in time for your event.
- Please send us your order number so we can check shipping options right away.

Example message:
I forgot my password and cannot get into my account.

Example output:
Urgency: medium
Topic: account
Suggested reply:
- Please use the password reset link on the sign-in page to create a new password.
- If that does not work, contact support and we can help verify your account.

Now analyze this customer message:
My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.


In [12]:
# Evaluate the few-shot prompt on the same test set.
few_shot_outputs = few_shot_chain.batch(baseline_inputs)
print_outputs(support_cases, few_shot_outputs)

Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
Urgency: high
Topic: product
Suggested reply:
- I'm sorry to hear the mug arrived broken and understand you need a replacement quickly.
- Please provide your order number and we will ship a new mug to you before Friday.
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
Urgency: medium
Topic: account
Suggested reply:
- Please use the password reset link on the sign‑in page to create a new password.
- If you encounter any issues, let us know and we’ll help verify your account.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
Urgency: medium
Topic: billing
Suggested reply:
- I’m sorry to hear you’ve been charged twice for your order.
- Please provide your order number and we’ll investigate the duplicate charge right away.
Case: Sustainability question
What to watch: Should stay low urgency and answer as

## Step 6 - Reuse the Prompt on a New Test Set

A prompt is more convincing when it generalizes beyond the first set of examples. Here we apply the same few-shot prompt to a second mini test set with different customer messages.

In [13]:
new_cases = [
    {
        "label": "Gift card request",
        "message": "Do you sell gift cards? I want to send one to a friend for their birthday.",
        "watch_for": "Should be product-related and low urgency.",
    },
    {
        "label": "Express shipping",
        "message": "I placed my order an hour ago. Can I upgrade it to express shipping?",
        "watch_for": "Should likely be shipping-related and time-sensitive.",
    },
]

new_outputs = few_shot_chain.batch([{"message": case["message"]} for case in new_cases])
print_outputs(new_cases, new_outputs)

Case: Gift card request
What to watch: Should be product-related and low urgency.
Urgency: low  
Topic: product  
Suggested reply:  
- Yes, we do sell digital gift cards that can be sent directly to your friend’s email.  
- You can purchase one from our gift card page and add a personalized birthday message.
Case: Express shipping
What to watch: Should likely be shipping-related and time-sensitive.
Urgency: medium  
Topic: shipping  
Suggested reply:  
- Thank you for reaching out; we can help you upgrade to express shipping if the order is still in processing.  
- Please provide your order number and we’ll check the status and apply the upgrade for you.


## Exercises

1. Add one more support case to `support_cases` and test all three prompt variants.
2. Change the improved prompt so it must return JSON-like output instead of bullet points.
3. Add one more few-shot example for a returns-related message and compare the result.
4. Raise the model temperature and check whether output consistency gets better or worse.


In [14]:
# Exercise answer scaffold: replace the message below with your own test case.
custom_case = {
    "message": "I returned my shoes last week but I still have not received the refund.",
}

print(few_shot_chain.invoke(custom_case))

Urgency: medium  
Topic: returns  
Suggested reply:  
- I’m sorry to hear that you haven’t received your refund yet.  
- Please provide your order number and the return tracking details so we can investigate the status and expedite the refund.


## Conclusion

In this notebook, you practiced a simple but powerful prompt-engineering workflow:
- define a task clearly
- create a compact evaluation set
- compare prompt variants on the same cases
- tighten the prompt format when outputs are inconsistent
- add few-shot examples when you need even more control

This kind of lightweight testing loop is a strong next step after learning the LangChain basics and the core prompt-engineering patterns.